In [279]:
import polars as pl
import options_data_utils
from importlib import reload
reload(options_data_utils)
from vol_utils import get_ohlc_data
from options_data_utils import get_cleaned_options_data
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
from format import format_chart
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
from format import format_chart


In [280]:
tickers_data = pl.read_csv('tickers.csv')
tickers = tickers_data['ticker'].to_list()

In [349]:
colors_and_category = {
        "Shiny Metals": '#DBAB61',
        "Large Cap Index": '#0F2573',
        "Mid Cap Index": '#B3D6F9',
        "Small Cap Index": '#3684DB',
        "Rates": '#294F50',
        "FX": '#B45F66',
        "US Credit": '#85BB65' ,
        "Energy": '#81A1C1',
        "Semi": '#4C566A',
        "Emerging Credit": '#a3be8c',
        "Emerging Index": '#C3C3C3',
        "Auto": '#8FBCBB'
    }
def color_provider (ticker,tickers_data=tickers_data, colors_and_category=colors_and_category):
    category = tickers_data.filter(pl.col('ticker') == ticker)['category'].item()   
    return colors_and_category[category]

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

def get_legend (colors_and_category=colors_and_category, **kwargs):
    # Create patches for the legend
    legend_elements = [Patch(facecolor=color, label=label) for label, color in colors_and_category.items()]
    # Create a new figure
    fig, ax = plt.subplots(figsize=(6, 2))
    # Create the legend
    ax.axis('off')

    legend = ax.legend(handles=legend_elements, loc='center', ncol=2, frameon=False)
    plt.setp(legend.get_texts(), font_properties=fm.FontProperties(fname='font/GeistMono.ttf'))

    return ax

get_legend()

In [308]:
covid=pl.col('trading_day')>datetime.date(2021,5,1)
delta_is_zero_or_one = (pl.col('delta').is_in([0, 1]))
def filter_out_crazy_vrps (self):
    return self.filter(self['vrp'].is_between(self['vrp'].quantile(0.0), self['vrp'].quantile(0.999)))
def housekeeping (ticker):
    df = pl.read_parquet(f"DATA/CLEANED_OPTIONS_DATA/CALL/{ticker}.parquet")
    df = df.filter(pl.col('underlying_price')!=0).filter(delta_is_zero_or_one.not_())
    df = df.with_columns(
        (pl.col('implied_vol') / pl.col('realized_volatility')).alias('vrp')
    ).filter_out_crazy_vrps()
    return df

import datetime



def get_atm_vrps (ticker):
    df = housekeeping(ticker).filter_atm_based_on_delta()
    return df

def get_otm_vrps (ticker):
    df = housekeeping(ticker).filter_otm_based_on_delta()
    return df


pl.DataFrame.filter_out_crazy_vrps = filter_out_crazy_vrps

In [ ]:
df = pl.read_parquet(f"DATA/CLEANED_OPTIONS_DATA/CALL/{ticker}.parquet").filter(pl.col('underlying_price')!=0)
df['trading_day'].unique().shape[0]

In [ ]:
@format_chart
def plot_vrp_time_series (tickers, **kwargs):
    fig, ax = plt.subplots(figsize=(15, 8))
    for ticker in tickers:
        df = get_atm_vrps(ticker).group_by('trading_day').agg(pl.col('vrp').mean()).sort('trading_day')
        ax.plot(df['trading_day'], df['vrp'], label=ticker)
        ax.legend()
    return ax
        
tickers = ['IEF','TLT','LQD','HYG', 'FXE','SPY', 'MDY','IWM','GLD', 'SLV',  'TSM', 'XLE' ]
plot_vrp_time_series(tickers)
   


In [ ]:
@format_chart
def plot_multi_violin(data_list, category_dict, xlabel, ylabel, title, figsize=(14, 8), horizontal_grid=False, vertical_grid=False):
    # Predefined color palette
    colors_and_category = {
        "Shiny Metals": '#DBAB61',
        "Large Cap Index": '#0F2573',
        "Mid Cap Index": '#B3D6F9',
        "Small Cap Index": '#3684DB',
        "Rates": '#294F50',
        "FX": '#B45F66',
        "US Credit": '#85BB65' ,
        "Energy": '#81A1C1',
        "Semi": '#4C566A',
        "Emerging Credit": '#a3be8c',
        "Emerging Index": '#C3C3C3',
        "Auto": '#8FBCBB'
    }
    
    # Create a DataFrame from the data list
    df = pd.DataFrame({name: pd.Series(array) for name, array in data_list})
    
    # Calculate means and sort columns by mean
    means = df.max().sort_values()
    df_sorted = df[means.index]
    
    # Melt the DataFrame to long format
    df_melted = df_sorted.melt(var_name='Distribution', value_name='Value')
    
    # Add category information
    df_melted['Category'] = df_melted['Distribution'].map(category_dict)

    fig, ax = plt.subplots(figsize=figsize)
    
    # Create the violin plot
    violin_parts = sns.violinplot(x='Distribution', y='Value', data=df_melted, ax=ax, width=1.2,
                   order=means.index, 
                   palette=[colors_and_category[category_dict[ticker]] for ticker in means.index],
                              )  # Shows quartile and median inside the violin
    

    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    for violin in violin_parts.collections:
        violin.set_alpha(0.75)
    # Rotate x-axis labels if there are many distributions
  
    
    # Add a legend
    handles = [plt.Rectangle((0,0),1,1, color=color) for category, color in colors_and_category.items()]
     # Calculate the number of columns for the legend
    n_categories = len(colors_and_category)
    ncols = (n_categories + 1) // 2  # This will give us 2 rows

    # Create the legend with two rows
    legend = plt.legend(handles, colors_and_category.keys(), title="Category", 
                        loc="lower center", 
                        bbox_to_anchor=(0.5, -0.2),  # Adjusted to make room for two rows
                        ncol=ncols,
                        columnspacing=1,  # Adjust spacing between columns
                        handletextpad=0.5)  # Adjust spacing between color boxes and text
    plt.setp(legend.get_texts(), font_properties=fm.FontProperties(fname='font/GeistMono.ttf'))
    plt.setp(legend.get_title(), font_properties=fm.FontProperties(fname='font/GeistMono.ttf', weight='bold'))

    plt.tight_layout()
    
    return ax




category_dict = dict(zip(tickers_data['ticker'], tickers_data['category']))

tickers = ['IEF','TLT','LQD','HYG', 'FXE','SPY', 'MDY','IWM','GLD', 'SLV',  'TSM', 'XLE' ]
plot_multi_violin([(ticker,get_atm_vrps(ticker).filter(pl.col('trading_day')>datetime.date(2022,1,1))['vrp']) for ticker in tickers], category_dict, xlabel='VRP', ylabel='Density', title='50-Delta VRP Distribution by Ticker', vertical_grid=True, figsize=(10, 11)) 

In [333]:
@format_chart
def plot_multi_violin_split(data_list, category_dict, xlabel, ylabel, title, figsize=(14, 8), horizontal_grid=False, vertical_grid=False):
    # Predefined color palette
    colors_and_category = {
        "Shiny Metals": '#DBAB61',
        "Large Cap Index": '#0F2573',
        "Mid Cap Index": '#B3D6F9',
        "Small Cap Index": '#3684DB',
        "Rates": '#294F50',
        "FX": '#B45F66',
        "US Credit": '#85BB65' ,
        "Energy": '#81A1C1',
        "Semi": '#4C566A',
    }
    
    # Create a DataFrame from the data list
    df = pd.DataFrame([(name, 'OTM VRP', val) for name, arr1, _ in data_list for val in arr1] +
                      [(name, 'ATM VRP', val) for name, _, arr2 in data_list for val in arr2],
                      columns=['Distribution', 'ArrayType', 'Value'])
    
    # Calculate means and sort distributions by mean of Array1
    means = df[df['ArrayType'] == 'OTM VRP'].groupby('Distribution')['Value'].mean().sort_values()
    df['Distribution'] = pd.Categorical(df['Distribution'], categories=means.index, ordered=True)
    
    # Add category information
    df['Category'] = df['Distribution'].map(category_dict)

    fig, ax = plt.subplots(figsize=figsize)
    
    # Create the split violin plot
    violin_parts = sns.violinplot(x='Distribution', y='Value', hue='ArrayType', 
                                  data=df, ax=ax, split=True, inner=None,
                                  order=means.index, 
                                  palette=['#FF0000', '#3684DB'])

    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    for violin in violin_parts.collections:
        violin.set_alpha(0.5)
    
    # Rotate x-axis labels if there are many distributions
    plt.xticks(rotation=45, ha='right')
    
    # Add a legend for categories
    handles = [plt.Rectangle((0,0),1,1, color=color) for category, color in colors_and_category.items()]
    n_categories = len(colors_and_category)
    ncols = (n_categories + 1) // 2  # This will give us 2 rows

    # Create the legend with two rows
    category_legend = plt.legend(handles, colors_and_category.keys(), title="Category", 
                        loc="lower center", 
                        bbox_to_anchor=(0.5, -0.4),  # Adjusted to make room for two rows
                        ncol=ncols,
                        columnspacing=1,  # Adjust spacing between columns
                        handletextpad=0.5)  # Adjust spacing between color boxes and text
    plt.setp(category_legend.get_texts(), font_properties=fm.FontProperties(fname='font/GeistMono.ttf'))
    plt.setp(category_legend.get_title(), font_properties=fm.FontProperties(fname='font/GeistMono.ttf', weight='bold'))

    # Add the category legend to the plot
    ax.add_artist(category_legend)

    # Adjust the position of the ArrayType legend
    array_type_legend = ax.legend(title="Array Type", loc="upper right")
    plt.setp(array_type_legend.get_texts(), font_properties=fm.FontProperties(fname='font/GeistMono.ttf'))
    plt.setp(array_type_legend.get_title(), font_properties=fm.FontProperties(fname='font/GeistMono.ttf', weight='bold'))

    plt.tight_layout()
    
    return ax

In [311]:
def get_data_list (ticker):
    name = ticker
    array1 = get_atm_vrps(ticker).filter(pl.col('trading_day')>datetime.date(2022,1,1))['vrp']
    array2 = get_otm_vrps(ticker).filter(pl.col('trading_day')>datetime.date(2022,1,1))['vrp']
    array2 = array2.filter(array2.is_between(0, 10))
    array1 = array1.filter(array1.is_between(0, 10))
    return name, array2, array1
    

In [ ]:
plot_multi_violin_split([get_data_list(ticker) for ticker in ['GLD', 'IEF', 'SPY']], category_dict, xlabel='VRP', ylabel='Density', title='ATM VRP Distribution by Ticker', vertical_grid=True, figsize=(11, 5)) 

In [313]:
def get_iv_data_list (ticker):    
    name = ticker
    array1 = get_atm_vrps(ticker).filter(pl.col('trading_day')>datetime.date(2022,1,1))['vrp']
    array2 = get_otm_vrps(ticker).filter(pl.col('trading_day')>datetime.date(2022,1,1))['vrp']
    array2 = array2.filter(array2.is_between(0, 10))
    array1 = array1.filter(array1.is_between(0, 10))
    return[ (name, array2), (name,array1)]

In [ ]:
plot_multiple_kde(get_iv_data_list('SPY'),  xlabel='VRP', ylabel='Density', title='ATM VRP Distribution by Ticker', figsize=(12, 8), override_colors=True)

In [ ]:
@format_chart
def plot_multiple_kde(data_list,  xlabel="Density", ylabel="Support", title="KDE", figsize=(12, 8),override_colors=False):  
    # Predefined color palette
    
    colors_and_category = {
            "Shiny Metals": '#DBAB61',
            "Large Cap Index": '#0F2573',
            "Mid Cap Index": '#B3D6F9',
            "Small Cap Index": '#3684DB',
            "Rates": '#294F50',
            "FX": '#B45F66',
            "US Credit": '#85BB65' ,
            "Energy": '#81A1C1',
            "Semi": '#4C566A',
        }
    
    fig, ax = plt.subplots(figsize=figsize)
    
    

    # Plot KDE for each array
    for i, (name, array) in enumerate(data_list):
        category = tickers_data.filter(pl.col('ticker') == name).select('category').item()
        color = colors_and_category.get(category, "#000000")  # Use black if color is not found
        if override_colors:
            colors = ['#0F2573', '#735D0F', '#294F50', '#502A29']
            sns.kdeplot(data=array, ax=ax, label=f"{name} ({category})", color=colors[i % len(colors)], fill=True, alpha=0.3)
        else:
            sns.kdeplot(data=array, ax=ax, label=f"{name} ({category})", color=color, fill=True, alpha=0.3)
    
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    
    # Add a legend
    # handles = [plt.Rectangle((0,0),1,1, color=color) for category, color in category_dict.items()]
    ax.legend(title="Category", loc="upper left", bbox_to_anchor=(1, 1))
    
    # Adjust layout to prevent clipping of labels and legend
    plt.tight_layout()
    
    return ax

plot_multiple_kde([(ticker, get_atm_vrps(ticker)['vrp']) for ticker in ['GLD', 'SPY', 'TLT', 'XLE']],  xlabel='VRP', ylabel='Density', title='50-Delta VRP Distribution by Ticker', figsize=(12, 8))

In [ ]:
plot_multiple_kde([(ticker, get_otm_vrps(ticker)['vrp']) for ticker in ['GLD', 'SPY', 'TLT', 'XLE']],  xlabel='VRP', ylabel='Density', title='25 to 30-Delta VRP Distribution by Ticker', figsize=(12, 8))

In [ ]:
plot_multiple_kde([(ticker, get_atm_vrps(ticker).filter(pl.col('trading_day')>datetime.date(2022,1,1))['vrp']) for ticker in ['IEF', 'TLT']],  xlabel='VRP', ylabel='Density', title='ATM VRP Distribution by Ticker', figsize=(12, 8), override_colors=True)

In [ ]:
category_dict = dict(zip(tickers_data['ticker'], tickers_data['category']))
plot_multi_violin([(ticker,get_otm_vrps(ticker)['vrp']) for ticker in tickers], category_dict, xlabel='VRP', ylabel='Density', title='25 to 30-Delta VRP Distribution by Ticker', figsize=(9,11), vertical_grid=True) 

In [347]:
def build_skew (ticker):
    atm = get_atm_vrps(ticker).select(['timestamp','realized_volatility', 'implied_vol']).group_by('timestamp').agg(pl.col('implied_vol').mean(), pl.col('realized_volatility').mean()).rename({'implied_vol': 'atm_implied_vol'})
    otm = get_otm_vrps(ticker).select(['timestamp','realized_volatility', 'implied_vol']).group_by('timestamp').agg(pl.col('implied_vol').mean()).rename({'implied_vol': 'otm_implied_vol'})
    skew = atm.join(otm, on='timestamp').sort('timestamp').with_columns((pl.col('otm_implied_vol')/pl.col('atm_implied_vol')).alias('skew'))
    return skew

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
from math import ceil, sqrt
import matplotlib.font_manager as fm

def plot_skew_vs_realized_vol(tickers):
    n = len(tickers)
    cols = ceil(sqrt(n))
    rows = ceil(n / cols)
    
    fig, axes = plt.subplots(rows, cols, figsize=(5*cols, 4*rows))
    fig.suptitle('Skew vs Realized Volatility for Each Ticker', fontsize=16, fontproperties=fm.FontProperties(fname='font/GeistMono.ttf'))
    
    if n == 1:
        axes = np.array([axes])
    axes = axes.flatten()
    
    geist_mono = fm.FontProperties(fname='font/GeistMono.ttf')
    
    for i, ticker in enumerate(tickers):
        
        skew = build_skew(ticker)
        ax = axes[i]
        ax.scatter(skew['realized_volatility'], skew['skew'], alpha=0.5,s=3.0, color=color_provider(ticker))
        ax.set_title(ticker, fontproperties=geist_mono)
        ax.set_xlabel('Realized Volatility', fontproperties=geist_mono)
        ax.set_ylabel('Skew (OTM IV / ATM IV)', fontproperties=geist_mono)
        
        corr = np.corrcoef(skew['realized_volatility'], skew['skew'])[0][1]
        ax.text(0.05, 0.95, f'Correlation: {corr:.2f}', transform=ax.transAxes, verticalalignment='top', fontproperties=geist_mono)
        
        # Set tick label font
        for label in ax.get_xticklabels() + ax.get_yticklabels():
            label.set_fontproperties(geist_mono)
    
    # Hide any unused subplots
    for j in range(i+1, len(axes)):
        axes[j].axis('off')
    
    plt.tight_layout()
    return fig

# Call the function
plot_skew_vs_realized_vol(tickers[6:])
plot_skew_vs_realized_vol(tickers[:6])

In [ ]:
plot_multi_violin([(ticker,build_skew(ticker)['skew']) for ticker in tickers], category_dict, xlabel='Skew', ylabel='Density', title='Skew Distribution by Ticker', vertical_grid=True, figsize=(10, 11)) 

In [328]:
def get_iv (ticker):
    return get_atm_vrps(ticker).select(['timestamp','realized_volatility', 'implied_vol']).group_by('timestamp').agg(pl.col('implied_vol').mean(), pl.col('realized_volatility').mean())

def get_iv_otm (ticker):
    return get_otm_vrps(ticker).select(['timestamp','realized_volatility', 'implied_vol']).group_by('timestamp').agg(pl.col('implied_vol').mean(), pl.col('realized_volatility').mean())

In [ ]:
def plot_implied_vs_realized_vol(tickers):
    n = len(tickers)
    cols = ceil(sqrt(n))
    rows = ceil(n / cols)
    
    fig, axes = plt.subplots(rows, cols, figsize=(5*cols, 4*rows))
    fig.suptitle('Implied vs Realized Volatility for Each Ticker', fontsize=16, fontproperties=fm.FontProperties(fname='font/GeistMono.ttf'))
    
    if n == 1:
        axes = np.array([axes])
    axes = axes.flatten()
    
    geist_mono = fm.FontProperties(fname='font/GeistMono.ttf')
    
    for i, ticker in enumerate(tickers):
        
        iv = get_iv(ticker)
        ax = axes[i]
        ax.scatter(iv['realized_volatility'], iv['implied_vol'], alpha=0.5,s=3.0, color=color_provider(ticker))
        ax.set_title(ticker, fontproperties=geist_mono)
        ax.set_xlabel('Realized Volatility', fontproperties=geist_mono)
        ax.set_ylabel('Implied Volatility', fontproperties=geist_mono)
        
        corr = np.corrcoef(iv['realized_volatility'], iv['implied_vol'])[0][1]
        ax.text(0.05, 0.95, f'Correlation: {corr:.2f}', transform=ax.transAxes, verticalalignment='top', fontproperties=geist_mono)
        
        # Set tick label font
        for label in ax.get_xticklabels() + ax.get_yticklabels():
            label.set_fontproperties(geist_mono)
    
    # Hide any unused subplots
    for j in range(i+1, len(axes)):
        axes[j].axis('off')
    
    plt.tight_layout()
    return fig

# Call the function
plot_implied_vs_realized_vol(tickers[6:])
plot_implied_vs_realized_vol(tickers[:6])

In [ ]:
def plot_implied_vs_realized_vol(tickers):
    n = len(tickers)
    cols = ceil(sqrt(n))
    rows = ceil(n / cols)
    
    fig, axes = plt.subplots(rows, cols, figsize=(5*cols, 4*rows))
    fig.suptitle('Implied vs Realized Volatility for Each Ticker', fontsize=16, fontproperties=fm.FontProperties(fname='font/GeistMono.ttf'))
    
    if n == 1:
        axes = np.array([axes])
    axes = axes.flatten()
    
    geist_mono = fm.FontProperties(fname='font/GeistMono.ttf')
    
    for i, ticker in enumerate(tickers):
        
        iv = get_iv_otm(ticker)
        ax = axes[i]
        ax.scatter(iv['realized_volatility'], iv['implied_vol'], alpha=0.5,s=3.0, color=color_provider(ticker))
        ax.set_title(ticker, fontproperties=geist_mono)
        ax.set_xlabel('Realized Volatility', fontproperties=geist_mono)
        ax.set_ylabel('Implied Volatility', fontproperties=geist_mono)
        
        corr = np.corrcoef(iv['realized_volatility'], iv['implied_vol'])[0][1]
        ax.text(0.05, 0.95, f'Correlation: {corr:.2f}', transform=ax.transAxes, verticalalignment='top', fontproperties=geist_mono)
        
        # Set tick label font
        for label in ax.get_xticklabels() + ax.get_yticklabels():
            label.set_fontproperties(geist_mono)
    
    # Hide any unused subplots
    for j in range(i+1, len(axes)):
        axes[j].axis('off')
    
    plt.tight_layout()
    return fig

# Call the function
plot_implied_vs_realized_vol(tickers[6:])
plot_implied_vs_realized_vol(tickers[:6])


In [276]:
import numpy as np
from scipy import stats

def sort_dict_by_trimmed_mean(data_dict, trim_percentage=10):
    """
    Sort a dictionary of numpy arrays by their trimmed mean.
    
    Args:
    data_dict (dict): Dictionary of numpy arrays to be sorted.
    trim_percentage (float): Percentage to trim from both ends of the distribution (default: 10).
    
    Returns:
    dict: Sorted dictionary of numpy arrays.
    """
    # Calculate trimmed mean for each array
    trimmed_means = {key: stats.trim_mean(arr, trim_percentage/100) for key, arr in data_dict.items()}
    
    # Sort the dictionary items based on their trimmed means (in descending order)
    sorted_items = sorted(data_dict.items(), key=lambda x: trimmed_means[x[0]], reverse=True)
    
    # Create a new sorted dictionary
    sorted_dict = dict(sorted_items)
    
    return sorted_dict

In [297]:

my_samples = {ticker: get_atm_vrps(ticker).to_numpy() for ticker in tickers}
t = my_samples 
# t  = sort_dict_by_trimmed_mean(my_samples)

In [ ]:
def sort_by_row_mean(arr):
    return arr[np.argsort(arr.mean(axis=1))[::-1]]
import numpy as np
from ridgeplot import ridgeplot


labels = tickers
fig = ridgeplot(samples=t.values(), labels=t.keys(),colorscale="viridis",
    colormode="row-index",coloralpha=0.65,)
fig.update_layout(height=1200, width=800)
fig.update_layout(
    title="Minimum and maximum daily temperatures in Lincoln, NE (2016)",
    height=700,
    width=1400,
    font_size=14,
    plot_bgcolor="rgb(245, 245, 245)",
    xaxis_gridcolor="white",
    yaxis_gridcolor="white",
    xaxis_gridwidth=2,
    yaxis_title="Ticker",
    xaxis_title="Volatility Risk Premium",
    showlegend=False,
)
fig.show()

In [220]:
from format import format_chart

@format_chart
def plot_vrp_against_strike (ticker):
    fig, ax = plt.subplots(figsize=(10, 8))
    f = pl.read_parquet(f"DATA/CLEANED_OPTIONS_DATA/CALL/{ticker}.parquet")
    f = f.with_columns(
    (pl.col('implied_vol') / pl.col('realized_volatility')).alias('vrp')
)
    cmap_name='Reds'
    # 
    cmap = cm.get_cmap(cmap_name)
    colors = cmap(rescale_to_01(f['realized_volatility'].to_numpy()))
    ax.scatter(x=f['strike'], y=f['implied_vol'], c=colors,  edgecolor="black", linewidth=0.5, s=30.0, alpha=rescale_to_01(f['days_to_expiry'].to_numpy()))
    ax.legend()
    return ax
    

In [ ]:
import holidays
my_holidays = holidays.country_holidays("US", years=range(2019, 2025))
call = call.with_columns(
    pl.col('timestamp').dt.add_business_days(-pl.col('days_to_expiry'), holidays=my_holidays, roll='backward').alias("offseted_time")
    )
call

In [15]:
df = call.filter((pl.col('trading_day') != pl.col('expiration'))).select('timestamp', 'offseted_time').unique().sort('timestamp').filter(pl.col('offseted_time')>ohlc['timestamp'].min())

In [ ]:
df

In [2]:
tickers = pl.read_csv('tickers.csv')['ticker'].to_list()

In [ ]:
import numpy as np
from functools import lru_cache
import holidays
from tqdm import tqdm
for ticker in tickers:
    print(ticker)
    call = get_cleaned_options_data(ticker)
    ohlc = get_ohlc_data(ticker)
    my_holidays = holidays.country_holidays("US", years=range(2019, 2025))
    call = call.with_columns(
    pl.col('timestamp').dt.add_business_days(-pl.col('days_to_expiry'), holidays=my_holidays, roll='backward').alias("offseted_time")
    )

    @lru_cache(maxsize=None)
    def get_realised_vol (options_data_row):
        upper = options_data_row[0]
        lower =  options_data_row[1]
        if lower < ohlc['timestamp'].min():
            print("Problem with ", lower, upper)
        else:
            try: 
                return ohlc.filter(pl.col('timestamp').is_between(lower, upper))['log_return_5m'].std() * np.sqrt(252*6.5*12)
            except:
                print("Problem with ", lower, upper)
    df = call.filter((pl.col('trading_day') != pl.col('expiration'))).select('timestamp', 'offseted_time').unique().filter(pl.col('offseted_time')>ohlc['timestamp'].min()).sort('timestamp')

    rvol = pl.Series(get_realised_vol(row) for row in tqdm(df.iter_rows(), total=len(df)))
    df = df.with_columns(
    rvol.alias('realized_volatility'))
    call.filter((pl.col('trading_day') != pl.col('expiration'))).filter(pl.col('offseted_time')>ohlc['timestamp'].min()).join(df, on=['timestamp','offseted_time'], how='left').write_parquet(f"DATA/CLEANED_OPTIONS_DATA/CALL/{ticker}.parquet")

In [ ]:
call

In [ ]:
call.filter().join(df, on='timestamp', how='left')

In [ ]:
get_realised_vol(ohlc, call)

In [ ]:
len(call.columns)

In [ ]:
call.join(ohlc, how='left', left_on='offseted_time', right_on='timestamp')